# 02 · RQ2 lead and lag

**Question.** Which platform tends to have a story first, and by how long? A platform that leads is an early-warning feed for the hours horizon ("should I post about this now?"), and its lead time bounds how early a forecast can be.

**Data.** The stored lists (`trend_items`) through RQ6's cross-platform matching (`topnews.echo`: the pipeline's own filters and model). Once topic snapshots exist, repeat this on them: each topic's first hour per platform.

**Method.** Matched trends (similarity at least 0.70, stricter than RQ6's 0.60 because a wrong match gives a meaningless lead) are grouped into stories. For each story and pair of platforms, the lead is the gap between each platform's earliest sighting. Three rules keep a lead honest:
- *censoring*: a trend already on its list at that source's first fetch started at an unknown earlier time, so its story pairs are left out (X, TikTok, Instagram and Pinterest started on 2026-09-30 22:08 UTC);
- *slow sources*: TikTok, Instagram and Pinterest appear on their refresh schedule, so they're left out;
- *resolution*: fetches are hourly, so leads are good to about an hour, and sightings under 30 minutes apart are a tie.

**Needs** `npm ci` at the repo root, as notebook 06 does. Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import db, echo, leadlag

pd.set_option("display.max_colwidth", 50)
engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out
starts = leadlag.source_starts(items)
cp, dropped = echo.analyze(items, name="rq2")
starts

## Matched pairs and stories

How many pairs there are, and how many survive censoring and the slow-source rule.

In [ ]:
pairs = leadlag.lead_pairs(cp, starts)
stories = leadlag.story_pairs(pairs)
usable = leadlag.usable(stories)
pd.DataFrame({
    "trend pairs": [len(pairs)],
    "stories": [stories.story.nunique() if len(stories) else 0],
    "story pairs": [len(stories)],
    "censored": [int(stories.censored.sum()) if len(stories) else 0],
    "slow": [int(stories.slow.sum()) if len(stories) else 0],
    "usable": [len(usable)],
})

## Who leads

Per pair of platforms (positive `median_lead_hours`: `platform_a` first), and per platform over all its usable story pairs.

In [ ]:
if usable.empty:
    print("No usable story pairs yet.")
else:
    display(leadlag.pair_summary(usable).round(2))
    display(leadlag.leaders(usable).round(2))

## Every story pair, for reading

Censored and slow pairs included, marked. Read the censored ones as "unknown", not as late: a platform that joined the data late can't show that it had a story earlier.

In [ ]:
if len(stories):
    view = stories.assign(first_a=stories.first_a.dt.strftime("%m-%d %H:%M"), first_b=stories.first_b.dt.strftime("%m-%d %H:%M"))
    display(view.drop(columns="story").sort_values(["censored", "slow", "label"]).round(2))

## How much data is on hand

Usable story pairs per pair of fast platforms. A lead per pair of platforms needs about 20 stories to say much; with a few cross-platform stories a day (RQ6), that takes weeks for most pairs.

In [ ]:
fast = sorted(set(cp.source_id) - leadlag.SLOW_SOURCES)
counts = usable.groupby(["platform_a", "platform_b"]).size() if len(usable) else pd.Series(dtype=int)
pd.DataFrame([[counts.get((a, b), 0) if a < b else None for b in fast] for a in fast], index=fast, columns=fast).astype("Int64")